# Tahap 3: Reranking (SBERT Similarity + Skor Efisiensi Bahan) — Chatbot Rekomendasi Resep

Lanjutan dari `embedding_indexing.ipynb`. Notebook ini:
1. Memuat ulang collection ChromaDB (`resep_masakan`) & model SBERT yang sudah dibuat sebelumnya
2. Mem-parsing input bahan dari user (raw text) jadi nama-nama bahan bersih
3. Mengambil kandidat resep dari ChromaDB (retrieval SBERT, top-K lebih besar dari hasil akhir)
4. Menghitung **skor efisiensi bahan** tiap kandidat: seberapa besar proporsi bahan resep yang sudah dimiliki user
5. Menggabungkan (blend) skor SBERT dengan skor efisiensi bahan jadi `final_score`, lalu rerank
6. Menyimpan daftar `missing_ingredients` per resep — dipakai nanti untuk alur konfirmasi "apakah ada bahan lain?" (tahap Llama3)
7. Perbandingan hasil top-5 SBERT-only vs top-5 setelah reranking (buat bahan evaluasi di skripsi)

> Catatan: notebook ini butuh folder `chroma_db` hasil dari `embedding_indexing.ipynb` sudah ada di direktori yang sama.

In [2]:
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path

pd.set_option("display.max_colwidth", 150)

# ==== KONFIGURASI (samakan dengan embedding_indexing.ipynb) ====
MODEL_NAME = "LazarusNLP/all-indo-e5-small-v4"
CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "resep_masakan"

CANDIDATE_K = 30  # jumlah kandidat yang diambil dari SBERT sebelum di-rerank
FINAL_TOP_N = 5  # jumlah resep final yang ditampilkan ke user
ALPHA = 0.65  # bobot skor SBERT vs skor efisiensi bahan (0-1). Makin besar -> makin condong ke similarity semantik
FUZZY_THRESHOLD = 80  # ambang skor fuzzy matching (0-100) untuk menangkap variasi penulisan nama bahan

## 1. Install Dependency Tambahan

In [7]:
# Jalankan sekali saja kalau rapidfuzz belum ke-install
!pip install -q rapidfuzz

## 2. Load Model SBERT & Collection ChromaDB

Load ulang model dan koneksi ke collection yang sudah di-index di `embedding_indexing.ipynb`. Tidak perlu encode ulang dataset.

In [4]:
import torch
from sentence_transformers import SentenceTransformer
import chromadb

device = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print("Device:", device)

model = SentenceTransformer(MODEL_NAME, device=device)

client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)
print("Collection:", collection.name, "| jumlah item:", collection.count())

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7371.10it/s]


Collection: resep_masakan | jumlah item: 5000


## 3. Parsing Input Bahan dari User

Reuse pendekatan regex yang sama seperti di `preprocessing.ipynb` (Bagian 7), supaya input user seperti `"100gr dada ayam"` diproses jadi nama bahan bersih `"dada ayam"` — konsisten dengan cara nama bahan resep di-parsing. Ini penting supaya matching di tahap efisiensi bahan apple-to-apple.

In [5]:
UNITS = [
    "kilogram",
    "kg",
    "gram",
    "grm",
    "gr",
    "g",
    "ons",
    "liter",
    "ltr",
    "sendok makan",
    "sdm",
    "sendok teh",
    "sdt",
    "ml",
    "l",
    "butir",
    "btr",
    "buah",
    "bh",
    "ekor",
    "siung",
    "lembar",
    "lbr",
    "batang",
    "btg",
    "ruas",
    "genggam",
    "ikat",
    "bungkus",
    "bgks",
    "papan",
    "potong",
    "ptg",
    "iris",
    "biji",
    "cm",
    "lonjor",
    "sisir",
    "sachet",
    "gelas",
    "cangkir",
    "pcs",
]
UNITS_SORTED = sorted(UNITS, key=len, reverse=True)
UNIT_PATTERN = "|".join(re.escape(u) for u in UNITS_SORTED)
QTY_PATTERN = r"(\d+[\.,]?\d*(?:\s*[/-]\s*\d+)?|setengah|seperempat|sepertiga)"

INGREDIENT_RE = re.compile(
    rf"^\s*(?P<qty>{QTY_PATTERN})?\s*(?P<unit>(?:{UNIT_PATTERN})(?![a-zA-Z]))?\.?\s*(?P<name>.*)$",
    flags=re.IGNORECASE,
)
PAREN_RE = re.compile(r"\(.*?\)")

SE_UNIT_MAP = {
    "sesendok": ("1", "sdm"),
    "secangkir": ("1", "cangkir"),
    "sebutir": ("1", "butir"),
    "seekor": ("1", "ekor"),
    "sesiung": ("1", "siung"),
    "selembar": ("1", "lembar"),
    "seruas": ("1", "ruas"),
    "segenggam": ("1", "genggam"),
    "sebungkus": ("1", "bungkus"),
    "sepotong": ("1", "potong"),
    "seiris": ("1", "iris"),
    "sebuah": ("1", "buah"),
    "sebatang": ("1", "batang"),
    "sesendokteh": ("1", "sdt"),
}
SE_UNIT_RE = re.compile(
    r"^(" + "|".join(sorted(SE_UNIT_MAP.keys(), key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)
SECUKUPNYA_RE = re.compile(r"^secukupnya\b", flags=re.IGNORECASE)


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_ingredient(raw: str) -> dict:
    text = clean_text(raw)
    text_no_paren = re.sub(r"\s+", " ", PAREN_RE.sub("", text)).strip()

    m = SECUKUPNYA_RE.match(text_no_paren)
    if m:
        name = text_no_paren[m.end() :].strip(" ,.-")
        return {
            "raw": text,
            "quantity": "secukupnya",
            "unit": None,
            "name": (name or text_no_paren).lower().strip(),
        }

    m = SE_UNIT_RE.match(text_no_paren)
    if m:
        key = m.group(1).lower()
        qty, unit = SE_UNIT_MAP[key]
        name = text_no_paren[m.end() :].strip(" ,.-")
        return {
            "raw": text,
            "quantity": qty,
            "unit": unit,
            "name": (name or text_no_paren).lower().strip(),
        }

    match = INGREDIENT_RE.match(text_no_paren)
    if not match:
        return {
            "raw": text,
            "quantity": None,
            "unit": None,
            "name": text_no_paren.lower().strip(),
        }
    qty = match.group("qty")
    unit = match.group("unit")
    name = match.group("name").strip(" ,.-")
    if not name:
        name = text_no_paren
    return {
        "raw": text,
        "quantity": qty.strip() if qty else None,
        "unit": unit.lower().strip() if unit else None,
        "name": name.lower().strip(),
    }


def parse_user_input_ingredients(text: str) -> list:
    """Pecah input user (dipisah koma) jadi list nama bahan bersih (tanpa qty/unit)."""
    parts = [p.strip() for p in re.split(r",|\n", text) if p.strip()]
    names = []
    for p in parts:
        parsed = parse_ingredient(p)
        if parsed["name"]:
            names.append(parsed["name"])
    return names


# contoh cepat
contoh = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"
print(parse_user_input_ingredients(contoh))

['telur', 'dada ayam', 'bawang putih']


## 4. Normalisasi & Matching Nama Bahan

Nama bahan hasil parsing (baik dari user maupun dari resep) kadang masih membawa kata deskriptif pengolahan (`cincang`, `iris`, `geprek`, `rebus`, dst) yang tidak relevan untuk matching. Fungsi di bawah membersihkan itu, lalu mencocokkan dua nama bahan lewat beberapa lapis:
1. **Exact match** setelah normalisasi
2. **Substring containment** (mis. `"bawang putih"` ada di dalam `"bawang putih cincang halus"`)
3. **Token overlap** (irisan kata yang sama, menangani urutan kata berbeda / bahan multi-kata)
4. **Fuzzy matching** (`rapidfuzz`) sebagai fallback untuk variasi penulisan/typo ringan

In [8]:
from rapidfuzz import fuzz

DESCRIPTOR_STOPWORDS = {
    "cincang",
    "cincangan",
    "halus",
    "iris",
    "irisan",
    "geprek",
    "memarkan",
    "dimemarkan",
    "potong",
    "dipotong",
    "potongan",
    "rebus",
    "direbus",
    "goreng",
    "digoreng",
    "bubuk",
    "segar",
    "matang",
    "mentah",
    "besar",
    "kecil",
    "tipis",
    "kasar",
    "cuci",
    "dicuci",
    "bersih",
    "sesuai",
    "selera",
    "kupas",
    "dikupas",
    "buang",
    "bijinya",
    "seratnya",
    "serong",
    "memanjang",
    "dadu",
    "utuh",
    "bulat",
    "muda",
    "tua",
    "kering",
    "basah",
    "sisir",
    "dibakar",
    "bakar",
    "parut",
    "diparut",
    "haluskan",
    "dihaluskan",
    "cair",
    "instan",
    "bakso",
    "sesuai",
    "selera",
    "ya",
    "yg",
    "yang",
    "sudah",
    "boleh",
    "bisa",
}
PUNCT_RE = re.compile(r"[^\w\s]")


def normalize_ingredient_name(name: str) -> str:
    if not isinstance(name, str):
        return ""
    text = name.lower()
    text = PUNCT_RE.sub(" ", text)
    tokens = [t for t in text.split() if t and t not in DESCRIPTOR_STOPWORDS]
    return " ".join(tokens).strip()


def is_ingredient_match(
    user_name_norm: str, recipe_name_norm: str, fuzzy_threshold: int = FUZZY_THRESHOLD
) -> bool:
    if not user_name_norm or not recipe_name_norm:
        return False
    if user_name_norm == recipe_name_norm:
        return True
    if user_name_norm in recipe_name_norm or recipe_name_norm in user_name_norm:
        return True
    user_tokens = set(user_name_norm.split())
    recipe_tokens = set(recipe_name_norm.split())
    if user_tokens and recipe_tokens:
        overlap = user_tokens & recipe_tokens
        smaller = min(len(user_tokens), len(recipe_tokens))
        if overlap and smaller > 0 and (len(overlap) / smaller) >= 0.5:
            return True
    if fuzz.token_set_ratio(user_name_norm, recipe_name_norm) >= fuzzy_threshold:
        return True
    return False


# sanity check cepat
tests = [
    ("bawang putih", "bawang putih cincang halus"),
    ("telur", "telur ayam"),
    ("dada ayam", "daging ayam fillet"),
    ("cabe merah", "cabai rawit"),
]
for a, b in tests:
    a_n, b_n = normalize_ingredient_name(a), normalize_ingredient_name(b)
    print(f"{a!r:20} vs {b!r:30} -> match={is_ingredient_match(a_n, b_n)}")

'bawang putih'       vs 'bawang putih cincang halus'   -> match=True
'telur'              vs 'telur ayam'                   -> match=True
'dada ayam'          vs 'daging ayam fillet'           -> match=True
'cabe merah'         vs 'cabai rawit'                  -> match=False


## 5. Skor Efisiensi Bahan

Untuk tiap kandidat resep, hitung berapa proporsi bahan resep yang match dengan bahan user (`matched_ingredients`), dan simpan juga daftar bahan yang **tidak** dimiliki user (`missing_ingredients`) — daftar ini yang nanti ditanyakan balik ke user di alur konfirmasi (tahap Llama3).

In [9]:
def compute_efficiency_score(
    user_ingredient_names: list, recipe_ingredient_names: list
):
    """
    Skor efisiensi = proporsi bahan RESEP yang match dengan bahan yang dimiliki user.
    Skor 1.0 berarti semua bahan resep sudah dimiliki user (paling efisien -> tidak perlu belanja tambahan).
    """
    if not recipe_ingredient_names:
        return 0.0, [], []

    user_norms = [normalize_ingredient_name(n) for n in user_ingredient_names]
    user_norms = [n for n in user_norms if n]

    matched, missing = [], []
    for r_raw in recipe_ingredient_names:
        r_norm = normalize_ingredient_name(r_raw)
        if any(is_ingredient_match(u, r_norm) for u in user_norms):
            matched.append(r_raw)
        else:
            missing.append(r_raw)

    score = len(matched) / len(recipe_ingredient_names)
    return score, matched, missing

## 6. Retrieval (SBERT) + Reranking (Blend Skor)

Alur:
1. Ambil `CANDIDATE_K` kandidat teratas dari ChromaDB berdasarkan similarity SBERT saja
2. Hitung `efficiency_score` tiap kandidat berdasarkan `ingredients_name_only` di metadata
3. Normalisasi `sbert_similarity` ke rentang [0, 1] **dalam himpunan kandidat ini** (min-max) supaya adil saat di-blend dengan `efficiency_score` yang sudah 0-1
4. `final_score = ALPHA * sbert_similarity_norm + (1 - ALPHA) * efficiency_score`
5. Urutkan berdasarkan `final_score`, ambil `FINAL_TOP_N` teratas

In [10]:
def retrieve_and_rerank(
    user_input_text: str,
    candidate_k: int = CANDIDATE_K,
    final_top_n: int = FINAL_TOP_N,
    alpha: float = ALPHA,
):
    user_ingredient_names = parse_user_input_ingredients(user_input_text)

    query_text = (
        "query: " + ", ".join(user_ingredient_names)
        if user_ingredient_names
        else "query: " + user_input_text
    )
    query_embedding = model.encode(
        [query_text], normalize_embeddings=True, convert_to_numpy=True
    ).tolist()

    results = collection.query(query_embeddings=query_embedding, n_results=candidate_k)

    rows = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        sim = (
            1 - results["distances"][0][i]
        )  # chroma cosine space -> distance = 1 - cosine_sim
        recipe_ingredient_names = json.loads(meta["ingredients_name_only"])
        eff_score, matched, missing = compute_efficiency_score(
            user_ingredient_names, recipe_ingredient_names
        )

        rows.append(
            {
                "id": results["ids"][0][i],
                "title": meta["title"],
                "sbert_similarity": sim,
                "efficiency_score": eff_score,
                "num_ingredients": meta["num_ingredients"],
                "matched_ingredients": matched,
                "missing_ingredients": missing,
                "ingredients_clean": json.loads(meta["ingredients_clean"]),
                "steps": json.loads(meta["steps"]),
            }
        )

    df_cand = pd.DataFrame(rows)

    sim_min, sim_max = (
        df_cand["sbert_similarity"].min(),
        df_cand["sbert_similarity"].max(),
    )
    if sim_max > sim_min:
        df_cand["sbert_similarity_norm"] = (df_cand["sbert_similarity"] - sim_min) / (
            sim_max - sim_min
        )
    else:
        df_cand["sbert_similarity_norm"] = 1.0

    df_cand["final_score"] = (
        alpha * df_cand["sbert_similarity_norm"]
        + (1 - alpha) * df_cand["efficiency_score"]
    )
    df_cand = df_cand.sort_values("final_score", ascending=False).reset_index(drop=True)

    return df_cand.head(final_top_n), user_ingredient_names, df_cand

## 7. Uji Coba — Sesuai Skenario di Skripsi

In [11]:
contoh_query = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"

hasil_final, user_names, df_semua_kandidat = retrieve_and_rerank(contoh_query)

print("Bahan user (setelah parsing):", user_names)
print()

for _, row in hasil_final.iterrows():
    print(f"[{row['title']}]")
    print(f"   sbert_similarity   : {row['sbert_similarity']:.4f}")
    print(
        f"   efficiency_score   : {row['efficiency_score']:.4f}  ({len(row['matched_ingredients'])}/{row['num_ingredients']} bahan cocok)"
    )
    print(f"   final_score        : {row['final_score']:.4f}")
    print(f"   bahan cocok        : {row['matched_ingredients']}")
    print(f"   bahan belum ada    : {row['missing_ingredients']}")
    print("-" * 90)

Bahan user (setelah parsing): ['telur', 'dada ayam', 'bawang putih']

[Dadar telur]
   sbert_similarity   : 0.7199
   efficiency_score   : 0.4000  (2/5 bahan cocok)
   final_score        : 0.7900
   bahan cocok        : ['telur', 'bawang merah']
   bahan belum ada    : ['garam', 'micin', 'merica']
------------------------------------------------------------------------------------------
[Telur Dadar Bawang]
   sbert_similarity   : 0.6770
   efficiency_score   : 0.5000  (3/6 bahan cocok)
   final_score        : 0.5524
   bahan cocok        : ['telur', 'tangkai daun bawang', 'bawang merah']
   bahan belum ada    : ['kaldu bubuk', 'garam', 'minyak goreng']
------------------------------------------------------------------------------------------
[Misua telur]
   sbert_similarity   : 0.6708
   efficiency_score   : 0.3333  (2/6 bahan cocok)
   final_score        : 0.4546
   bahan cocok        : ['telur', 'sawi putih']
   bahan belum ada    : ['gula', 'garam', 'lada', 'misua']
--------------

## 8. Perbandingan: SBERT-only vs Setelah Reranking

Menampilkan top-5 kalau HANYA pakai similarity SBERT (tanpa efisiensi bahan) berdampingan dengan top-5 hasil reranking. Berguna sebagai bahan analisis/evaluasi di bab hasil skripsi — untuk menunjukkan reranking benar-benar mengubah urutan ke arah resep yang lebih hemat bahan tambahan.

In [12]:
top_sbert_only = df_semua_kandidat.sort_values(
    "sbert_similarity", ascending=False
).head(FINAL_TOP_N)
top_reranked = df_semua_kandidat.sort_values("final_score", ascending=False).head(
    FINAL_TOP_N
)

perbandingan = pd.DataFrame(
    {
        "rank": range(1, FINAL_TOP_N + 1),
        "SBERT-only (title)": top_sbert_only["title"].values,
        "SBERT-only (sim)": top_sbert_only["sbert_similarity"].round(4).values,
        "Reranked (title)": top_reranked["title"].values,
        "Reranked (final_score)": top_reranked["final_score"].round(4).values,
        "Reranked (bahan cocok)": [
            f"{len(m)}/{n}"
            for m, n in zip(
                top_reranked["matched_ingredients"], top_reranked["num_ingredients"]
            )
        ],
    }
)
perbandingan

,rank,SBERT-only (title),SBERT-only (sim),Reranked (title),Reranked (final_score),Reranked (bahan cocok)
0,1,Dadar telur,0.7199,Dadar telur,0.7900,2/5
1,2,Telur Dadar Bayam 🥚💚,0.6793,Telur Dadar Bawang,0.5524,3/6
2,3,Telur Dadar Bawang,0.6770,Misua telur,0.4546,2/6
3,4,Misua telur,0.6708,Telur Dadar Bayam 🥚💚,0.4497,1/6
4,5,Telur Dadar Toge,0.6619,Telur Dadar Toge,0.4364,4/9


## 9. Coba Beberapa Skenario Query Lain

Termasuk skenario ketika user cuma menyebutkan sedikit bahan utama saja (sesuai use case chatbot kamu).

In [13]:
skenario_queries = [
    "telur, nasi putih, kecap manis",
    "ikan lele, cabe rawit, tomat",
    "tahu, tempe, santan",
]

for q in skenario_queries:
    hasil, user_names, _ = retrieve_and_rerank(q, final_top_n=3)
    print(f"=== Query: '{q}' -> bahan terparsing: {user_names} ===")
    for _, row in hasil.iterrows():
        print(
            f"  - {row['title']}  (final_score={row['final_score']:.4f}, cocok={len(row['matched_ingredients'])}/{row['num_ingredients']}, kurang={row['missing_ingredients']})"
        )
    print()

=== Query: 'telur, nasi putih, kecap manis' -> bahan terparsing: ['telur', 'nasi putih', 'kecap manis'] ===
  - Nasi telur kecap  (final_score=0.7900, cocok=4/10, kurang=['daun bawang', 'saus tiram', 'minyak wijen', 'saus sambal', 'lada bubuk', 'minyak goreng'])
  - Telur Kecap USA  (final_score=0.4620, cocok=3/7, kurang=['air', 'bawang bombay', 'daun bawang', 'minyak makan'])
  - Misua telur  (final_score=0.4440, cocok=2/6, kurang=['gula', 'garam', 'lada', 'misua'])

=== Query: 'ikan lele, cabe rawit, tomat' -> bahan terparsing: ['ikan lele', 'cabe rawit', 'tomat'] ===
  - Lele Cabe Ijo  (final_score=0.7812, cocok=3/8, kurang=['jeruk nipis', 'bawang merah', 'bawang putih', 'gula, garem, kaldu ayam', 'minyak utk menggoreng dan menumis'])
  - Oseng lele cabe rawit  (final_score=0.5374, cocok=3/14, kurang=['baput', 'bamer', 'jahe', 'lengkuas', 'daun sereh', 'daun salam', 'daun jeruk', '1/2 sendok makan garam', 'sedok teh micin', 'royco', 'bawang bombay'])
  - Sambal ikan Roa  (final_scor

## Catatan & Langkah Selanjutnya

1. **Tuning `ALPHA`** — coba beberapa nilai (mis. 0.5, 0.65, 0.8) dan lihat pengaruhnya ke urutan hasil. Kalau mau lebih condong ke "yang paling hemat bahan", turunkan `ALPHA`; kalau mau lebih condong ke "paling relevan secara semantik/topik", naikkan `ALPHA`. Ini juga bisa jadi salah satu eksperimen di bab evaluasi.
2. **Tuning `FUZZY_THRESHOLD` dan `DESCRIPTOR_STOPWORDS`** — kalau nanti ketemu kasus nama bahan yang gagal match padahal harusnya sama (atau sebaliknya, salah match), catat contohnya dan kita perbaiki daftar stopword / ambang fuzzy-nya.
3. **Alur konfirmasi bahan pelengkap (belum ada di notebook ini)** — `missing_ingredients` pada tiap hasil di atas adalah input utama untuk langkah ini: sebelum menampilkan resep final, AI (Llama3) bisa bertanya ke user, misalnya *"Selain telur, dada ayam, dan bawang putih, apakah kamu juga punya [daftar dari missing_ingredients yang paling umum, mis. garam/minyak goreng]?"* — lalu jawaban user dipakai untuk update `user_ingredient_names` dan panggil ulang `retrieve_and_rerank()`.
4. **Evaluasi retrieval** — kalau skripsi butuh metrik kuantitatif (Precision@K, NDCG, dst), kita perlu bikin dulu semacam ground-truth/relevance judgment (manual anotasi sebagian query), baru bisa dihitung. Bilang aja kalau bagian ini mau dikerjakan berikutnya.